In [4]:
%cd ../..

/gpfs/data/schambralab/quantitativeRehabilitation/__lab_member_homes/victor/cvfm4rehab


In [5]:

from __future__ import annotations

import json
import re
from pathlib import Path
from typing import Any, Iterable, Set

import numpy as np
import pandas as pd

from data.utils_strokerehab import DataPaths

In [30]:
from __future__ import annotations

import json
import re
from pathlib import Path
from typing import Iterable, Tuple, List

import numpy as np
import pandas as pd


# ─────────────────────────── helpers ─────────────────────────── #

SEP_RE   = re.compile(r"\s*<SEP>\s*")
RESP_RE  = re.compile(r"\s*<RESP>\s*")
TIME_RE  = re.compile(r"<TIME>\s*([\d.]+)-([\d.]+)", re.IGNORECASE)


def _parse_filtered_resps(raw: str | list[str]) -> List[List[Tuple[str, float, float]]]:
    """
    Split `filtered_resps` into blocks.
    Returns list[ block ], where each *block* is list[ (answer, t_start, t_end) ]
    matching the QID order.
    """
    if isinstance(raw, list):
        raw = "".join(raw)

    blocks = []
    for chunk in RESP_RE.split(raw):
        chunk = chunk.strip()
        if not chunk:
            continue

        # grab times first
        m = TIME_RE.search(chunk)
        if not m:
            raise ValueError("Missing <TIME …> tag in block.")
        t_start, t_end = map(float, m.groups())

        # strip the time portion
        answers_part = TIME_RE.sub("", chunk).strip()
        answers = [a.strip() for a in SEP_RE.split(answers_part) if a.strip()]
        blocks.append([(ans, t_start, t_end) for ans in answers])

    return blocks


def _calc_answer(qid: int, block_list: List[Tuple[str, float, float]]) -> str:
    """
    Given all blocks for one QID, produce the final `answer` string.
    """
    # 0‒94  → echo raw sequence
    if qid < 95:
        return block_list[0][0] if block_list else pd.NA

    # 95,96 → average & round
    if qid in (95, 96):
        nums = [float(ans) for ans, _, _ in block_list]
        return str(int(round(np.mean(nums)))) if nums else pd.NA

    # 97-100 → time when cumulative count hits threshold
    threshold = {97: 4, 98: 5, 99: 4, 100: 5}.get(qid)
    if threshold is not None:
        cum = 0
        for ans, _, t_end in block_list:
            try:
                cum += int(float(ans))
            except ValueError:
                continue
            if cum >= threshold:
                return t_end
        return np.inf

    # fallback
    return pd.NA


# ───────────────────────── main function ─────────────────────── #

def extract_answers(
    output_log_path: str | Path | Iterable[str | Path],
    drop_parsed: bool = True
) -> pd.DataFrame:
    """
    Returns DataFrame[patient, qid, answer, parsed_response].
    * parsed_response includes answer *and* start-/end-times joined by " | ".
    * answer is filled per rules above.
    """
    paths = (
        [output_log_path]
        if isinstance(output_log_path, (str, Path))
        else list(output_log_path)
    )

    # pass 1 – gather all patients & qids
    patients, qids = set(), set()
    for p in paths:
        with Path(p).open(encoding="utf-8") as fh:
            for line in fh:
                rec = json.loads(line)
                patients.add(rec["doc"]["patient"])
                qids.update(int(x) for x in rec["qids"].split("<SEP>"))

    full_idx = pd.MultiIndex.from_product(
        [sorted(patients), sorted(qids)], names=["patient", "qid"]
    )
    df = pd.DataFrame(index=full_idx).reset_index()
    df["answer"] = pd.NA
    df["parsed_response"] = pd.NA

    filled_pairs = set()

    # pass 2 – fill parsed_response & answer
    for p in paths:
        with Path(p).open(encoding="utf-8") as fh:
            for line in fh:
                rec = json.loads(line)
                patient = rec["doc"]["patient"]
                qids_line = [int(x) for x in rec["qids"].split("<SEP>")]

                blocks = _parse_filtered_resps(rec["filtered_resps"])
                if any(len(b) != len(qids_line) for b in blocks):
                    raise ValueError(f"Answer/QID mismatch for patient={patient}")

                # transpose: For each qid gather its per-block tuples
                per_qid: List[List[Tuple[str, float, float]]] = [
                    [] for _ in qids_line
                ]
                for blk in blocks:
                    for idx, triple in enumerate(blk):
                        per_qid[idx].append(triple)

                for idx, qid in enumerate(qids_line):
                    key = (patient, qid)
                    if key in filled_pairs:
                        raise ValueError(f"Duplicate entry for {key}")
                    triples = per_qid[idx]

                    # build parsed_response string
                    parsed = " | ".join(
                        f"{ans} {ts:.3f}-{te:.3f}" for ans, ts, te in triples
                    )

                    # compute answer
                    ans = _calc_answer(qid, triples)

                    mask = (df["patient"] == patient) & (df["qid"] == qid)
                    df.loc[mask, ["parsed_response", "answer"]] = [parsed, ans]
                    filled_pairs.add(key)

    if drop_parsed:
        df.drop(columns=["parsed_response"], inplace=True)
    return df


In [31]:
import re
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd

from data.utils_strokerehab import DataPaths   # keep your existing import

# ────────────────────────── helpers ────────────────────────── #

def _score_single_row(row: pd.Series) -> tuple[bool, int | None]:
    """
    Decide whether this row yields a *predicted* score.

    Returns
    -------
    got_score : bool
    score     : int | None
    """
    if pd.isna(row.answer):
        return False, None

    ans = str(row.answer).lower().strip()

    if row.question_type == "binary":
        if "yes" in ans and pd.notna(row.binary_yes_score):
            return True, int(row.binary_yes_score)
        if "no" in ans and pd.notna(row.binary_no_score):
            return True, int(row.binary_no_score)
        return False, None

    # rate-type questions (expects 0/1/2 somewhere in the string)
    m = re.search(r"\b([012])\b", ans)
    return (True, int(m.group(1))) if m else (False, None)


# ──────────────────────── main routine ─────────────────────── #

def compute_fm_scores_from_qids(
    ans_df: pd.DataFrame,
    *,
    questions_csv_path: str | Path = DataPaths.IA_QUESTIONS_PATH,
    gt_csv_path: str | Path = DataPaths.IA_SCORES_PATH,
    side_col: str = "Side of body affected",
    id_col: str = "Subject ID",
) -> pd.DataFrame:
    """
    Convert the *answer* DataFrame (from `extract_answers`) into predicted FM
    scores and attach ground-truth scores.

    Returns
    -------
    DataFrame with columns
        patient | fm_item | pred_score | gt_score
    """

    # ── 1) bring in question metadata ───────────────────────── #
    qmeta = pd.read_csv(
        questions_csv_path,
        usecols=[
            "qid",
            "fm_video",
            "question_type",
            "binary_no_score",
            "binary_yes_score",
        ],
    )
    qmeta["fm_item"] = qmeta["fm_video"].str.split("_").str[0].astype(int)

    merged = ans_df.merge(qmeta, on="qid", how="left")

    # ── 2) per-item prediction ─────────────────────────────── #
    scored_rows: list[dict[str, Any]] = []
    for (patient, fm_item), grp in merged.groupby(["patient", "fm_item"]):
        if fm_item == 33:                     # item-33 not scored
            # pull answers (floats) from the four timing questions
            a_vals = grp.loc[grp["qid"].isin([97, 98]), "answer"].dropna()
            b_vals = grp.loc[grp["qid"].isin([99, 100]), "answer"].dropna()

            a = float(np.nanmin(a_vals.astype(float))) if not a_vals.empty else np.inf
            b = float(np.nanmin(b_vals.astype(float))) if not b_vals.empty else np.inf

            if np.isinf(a) or np.isinf(b):
                score = np.nan       # insufficient data
            else:
                diff = a - b
                if diff < 2:
                    score = 2
                elif diff < 6:
                    score = 1
                else:
                    score = 0
        else:
            grp = grp.sort_values("qid")
            score = next(
                (s for got, s in (_score_single_row(r) for _, r in grp.iterrows()) if got),
                np.nan,
            )
        scored_rows.append(
            {"patient": patient, "fm_item": fm_item, "pred_score": score}
        )

    df = (
        pd.DataFrame(scored_rows)
        .sort_values(["patient", "fm_item"])
        .reset_index(drop=True)
    )

    # ── 3) infer FM-18 from 15-17 if missing ───────────────── #
    new_rows: list[dict[str, Any]] = []
    for patient, grp in df.groupby("patient"):
        if (grp["fm_item"] == 18).any():
            continue

        scores_15_17 = grp.loc[grp["fm_item"].isin([15, 16, 17]), "pred_score"]
        all_two = (len(scores_15_17) == 3) and ((scores_15_17 == 2).all())
        new_rows.append(
            {"patient": patient, "fm_item": 18, "pred_score": 2 if all_two else 0}
        )

    if new_rows:
        df = (
            pd.concat([df, pd.DataFrame(new_rows)], ignore_index=True)
            .sort_values(["patient", "fm_item"])
            .reset_index(drop=True)
        )

    df["pred_score"] = df["pred_score"].astype("Int64")

    # ── 4) attach ground-truth ──────────────────────────────── #
    gt = pd.read_csv(gt_csv_path).set_index(id_col)

    def _lookup_gt(row: pd.Series) -> int | float:
        pid = row.patient
        if pid not in gt.index:
            return np.nan
        side_suffix = "R" if str(gt.at[pid, side_col]).strip().capitalize() == "Right" else "L"
        col = f"{row.fm_item}{side_suffix}"
        return gt.at[pid, col] if col in gt.columns else np.nan

    df["gt_score"] = df.apply(_lookup_gt, axis=1).astype("Int64")
    return df


In [32]:
log_path_1 = "logs/strokerehab_ia_1/bot_8f/20250808_050104_samples_strokerehab_ia_1.jsonl"
log_path_2 = "logs/strokerehab_ia_2/bot_8f/20250808_050123_samples_strokerehab_ia_2.jsonl"
ans_df = extract_answers([log_path_1, log_path_2])

In [33]:
ans_df

,patient,qid,answer
0,S0001,0,2
1,S0001,1,2
2,S0001,2,2
3,S0001,3,2
4,S0001,4,2
...,...,...,...
96,S0001,96,2
97,S0001,97,0.533
98,S0001,98,0.8
99,S0001,99,0.533


In [34]:
ans_df

,patient,qid,answer
0,S0001,0,2
1,S0001,1,2
2,S0001,2,2
3,S0001,3,2
4,S0001,4,2
...,...,...,...
96,S0001,96,2
97,S0001,97,0.533
98,S0001,98,0.8
99,S0001,99,0.533


In [35]:
scores  = compute_fm_scores_from_qids(ans_df)


In [36]:
scores

,patient,fm_item,pred_score,gt_score
0,S0001,3,2,2
1,S0001,4,2,2
2,S0001,5,2,2
3,S0001,6,2,2
4,S0001,7,2,2
5,S0001,8,2,1
6,S0001,9,2,2
7,S0001,10,2,1
8,S0001,11,2,2
9,S0001,12,2,2


In [37]:
# --------------------------------------------------------------------------- #
# 3.  Aggregate evaluation metrics                                           #
# --------------------------------------------------------------------------- #


def _parse_patients(spec: str | None) -> Set[str] | None:
    if spec is None or not str(spec).strip():
        return None
    return {p.strip() for p in spec.split(",") if p.strip()}


def _parse_items(spec: str | None) -> Set[int] | None:
    if spec is None or not str(spec).strip():
        return None
    out: set[int] = set()
    for tok in spec.split(","):
        tok = tok.strip()
        if m := re.fullmatch(r"(\d+)-(\d+)", tok):
            start, end = map(int, m.groups())
            out.update(range(start, end + 1))
        elif tok.isdigit():
            out.add(int(tok))
        else:
            raise ValueError(f"Unparsable fm_item token: {tok!r}")
    return out


def _all_items_from_csv(csv_path: str | Path) -> Set[int]:
    qmeta = pd.read_csv(csv_path, usecols=["fm_video"])
    return {int(x.split("_")[0]) for x in qmeta["fm_video"]}


def aggregate_fm_metrics(
    score_df: pd.DataFrame,
    questions_csv_path: str | Path = DataPaths.IA_QUESTIONS_PATH,
    *,
    fm_items: str | None = None,
    patients: str | None = None,
) -> dict[str, float]:
    """
    Compute Accuracy, Average-Patient-Deviation (APD) and
    Mean-Total-Score-Deviation (MTSD) for (optionally) selected
    items and/or patients.
    """
    # 1) Item & patient subset
    item_set = _parse_items(fm_items) or _all_items_from_csv(questions_csv_path)
    patient_set = set(score_df["patient"].unique())
    if patients is not None:
        patient_set &= _parse_patients(patients)  # intersection

    if not item_set:
        raise ValueError("Item subset is empty.")
    if not patient_set:
        raise ValueError("Patient subset is empty.")

    # 2) Slice & ensure full grid
    df = score_df.loc[
        score_df["patient"].isin(patient_set) & score_df["fm_item"].isin(item_set),
        ["patient", "fm_item", "pred_score", "gt_score"],
    ]
    full_idx = pd.MultiIndex.from_product(
        [sorted(patient_set), sorted(item_set)],
        names=["patient", "fm_item"],
    )
    df = (
        df.set_index(["patient", "fm_item"])
        .reindex(full_idx)
        .reset_index()
    )

    pred, gt = df["pred_score"], df["gt_score"]

    # 3) Accuracy
    accuracy = float((pred == gt).sum(min_count=1)) / len(df)

    # 4) APD  (missing pred → error 2)
    df["err"] = [
        2 if pd.isna(p) else abs(int(p) - int(g)) for p, g in zip(pred, gt)
    ]
    apd = df.groupby("patient")["err"].sum().mean()

    # 5) MTSD  (patient-level totals + penalty for misses)
    agg = df.groupby("patient").agg(
        pred_sum=("pred_score", lambda s: s.fillna(0).sum()),
        gt_sum=("gt_score", "sum"),
        n_missing=("pred_score", lambda s: s.isna().sum()),
    )
    agg["patient_diff"] = (
        (agg["pred_sum"] - agg["gt_sum"]).abs() + 2 * agg["n_missing"]
    )
    mtsd = agg["patient_diff"].mean()

    return {"accuracy": accuracy, "apd": apd, "mtsd": mtsd}


In [38]:
aggregate_fm_metrics(scores)

{'accuracy': 0.5333333333333333, 'apd': 18.0, 'mtsd': 8.0}